# API 调用与检索

用设备说明试关键词检索和语义检索，再查看基于资料的回答。函数整理在 `main.py`。语义检索部分可以单独运行，不需要前面的 DeepSeek 客户端。

In [ ]:
from main import records, create_client, ask_model, find_records, build_context, answer_question

## 直接提问

输入密钥后修改问题试一下。运行下一格会请求 DeepSeek。

In [ ]:
client = create_client()
question = "Python列表和字典有什么区别？"
print(ask_model(question, None, client))

## 先看检索结果

检索返回记录列表，整理后得到保留来源的字符串。这格不请求模型。

In [ ]:
hits = find_records(records, "风机")
context = build_context(hits)
print(context)

## 根据找到的资料回答

分别检查：有答案、资料没写答案、没有相关资料。前两组会请求模型，最后一组直接返回。

In [ ]:
cases = [
    ("多久检查风机滤网？", "滤网"),
    ("风机的生产厂家是谁？", "风机"),
    ("电机的额定功率是多少？", "电机"),
]
for question, keyword in cases:
    print("问题：", question)
    print("回答：", answer_question(records, question, keyword, client))
    print()

## 换个关键词

‘过滤网’在原文中没有完整出现。这种查找无法识别它和‘滤网’的关系。

In [ ]:
print(find_records(records, "过滤网"))
assert find_records(records, "") == []
assert build_context([]) == ""

## 语义检索

用 BGE 中文模型把四段原文编码成向量。首次运行会下载模型，使用 CPU；后面几格只检索，不请求 DeepSeek。

In [ ]:
from main import records, load_embedder, semantic_search, get_records

model = load_embedder()
texts = [record["text"] for record in records]
doc_vectors = model.encode(texts, normalize_embeddings=True)
print(doc_vectors.shape)

### 用‘过滤网’找‘滤网’

问题和原文编码后比较余弦相似度。`corpus_id` 用来取回原始记录，`score` 用来排序。

In [ ]:
question = "过滤网多久维护一次？"
hits = semantic_search(records, question, model, doc_vectors, top_k=2)
for hit, record in zip(hits, get_records(records, hits)):
    print(f"{hit['score']:.4f} | {record['source']} | {record['text']}")

### 换三个问题

观察段落里是否包含答案。资料库非空时，取前两条通常仍会得到结果；相似度高不代表原文一定能回答问题。

In [ ]:
questions = [
    "机器太热应该怎么处理？",
    "水泵密封多久检查一次？",
    "设备的生产厂家是谁？",
]
for question in questions:
    hits = semantic_search(records, question, model, doc_vectors)
    print("问题：", question)
    for record in get_records(records, hits):
        print(record["source"], record["text"])
    print()

assert get_records(records, []) == []